# JRDB RA-L visualization notebook

Self-contained JRDB-only notebook for two conceptual manuscript figures:

1. **Real JRDB evaluation-regime overview** — one horizontal BEV scene with a focal pedestrian, matched Person-MinkUNet detections, optional surrounding GT pedestrian motion, and a simplified **non-overlapping visual segmentation** into Initialize / Continuous tracking / Gap coasting.
2. **Pseudo-detection failure modes** — Clean / Dropout / Instability / Combined, where Combined is literally the same instability realization with the same dropout mask applied.

## Coordinate-frame convention

The real-data figure deliberately uses two coordinate frames for two different purposes:

- **Local JRDB/base frame:** raw local GT and raw detector JSON are matched here with the same 0.30 m augmented-Hungarian center-distance rule as the evaluation-side detector-support cache.
- **Global frame:** stored `gt__global` boxes are used for GT trajectories/context. The matched raw detector box is transformed with the same SteamLO convention as `tracker_eval` before visualization.
- A final constant **plot-only rotation** is applied to every global object so the focal trajectory runs approximately left-to-right. This does not affect distances or matching.

## Visual-regime simplification

The arrows in Figure 1 are intentionally a **schematic, non-overlapping visualization**, not a literal reconstruction of all manuscript evaluation populations:

- **(a) Initialize:** 0.5 s beginning at the first matched detector support in the selected trajectory view, and again after a bounded detector gap longer than 1.5 s.
- **(b) Continuous tracking:** detector-supported intervals connected across gaps of at most 3 missing frames, excluding any Initialize portion at the interval start.
- **(c) Gap coasting:** the missing frames of every bounded detector gap longer than 3 frames.

The quantitative manuscript analyses remain defined by `tracker-eval-protocol`; this figure is a simplified explanatory view.

## Caching

Expensive raw GT↔detector frame matching and the all-sequence candidate scan are cached automatically under `CACHE_DIR`. Set `OVERWRITE_CACHE = True` to force recomputation. Figure files have their own `OVERWRITE_FIGURES` switch.


In [ ]:
# Cell 1 — Imports + configuration
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, FancyArrowPatch, Polygon
from scipy.optimize import linear_sum_assignment
from scipy.signal import savgol_filter
from IPython.display import display

# -----------------------------------------------------------------------------
# Paths — override these portable defaults with environment variables.
# -----------------------------------------------------------------------------
JRDB_LOCAL_GT_FOLDER = Path(
    os.environ.get(
        "JRDB_LOCAL_GT_FOLDER",
        "./trackeval_layout/data/gt/jrdb/jrdb_3d_box_test/label_02",
    )
).expanduser()

JRDB_GLOBAL_GT_FOLDER = Path(
    os.environ.get(
        "JRDB_GLOBAL_GT_FOLDER",
        "./trackeval_layout/data/gt__global/jrdb/jrdb_3d_box_test/label_02",
    )
).expanduser()

JRDB_DETECTIONS_DIR = Path(
    os.environ.get(
        "JRDB_DETECTIONS_DIR",
        "./data/test/detections_3d",
    )
).expanduser()

# Directly .../test/odometry: one <sequence>.csv per sequence.
JRDB_ODOMETRY_DIR = Path(
    os.environ.get(
        "JRDB_ODOMETRY_DIR",
        "./data/test/odometry",
    )
).expanduser()

FIGURE_OUTPUT_DIR = Path(
    os.environ.get("JRDB_RAL_VIS_OUTPUT_DIR", "./ral_visualization_figures")
).expanduser()
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = Path(
    os.environ.get("JRDB_RAL_VIS_CACHE_DIR", str(FIGURE_OUTPUT_DIR / "_cache"))
).expanduser()
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# False = reuse compatible cached results automatically.
# True = recompute cached GT↔detector matches and candidate scan.
OVERWRITE_CACHE = False
# Controls only writing PDF outputs; figures are still constructed/displayed.
OVERWRITE_FIGURES = True

# -----------------------------------------------------------------------------
# Shared evaluation settings
# -----------------------------------------------------------------------------
FPS = 15.0
DET_MATCH_MAX_DIST_M = 0.30
# Keep None to reproduce the current detector-support evaluation if the JSON
# export is already thresholded. Set 0.5 only if an extra plotting-time filter
# is deliberately desired.
DET_SCORE_MIN = None

# -----------------------------------------------------------------------------
# Figure 1: selected real JRDB trajectory
# -----------------------------------------------------------------------------
# SEQUENCE = "food-trucks-2019-02-12_0"
# SELECTED_GT_ID = 42
# FRAME_START = 850
# FRAME_END = 1020

SEQUENCE = "huang-intersection-2019-01-22_0"
SELECTED_GT_ID = 52
FRAME_START = 835
FRAME_END = 1000

# Simplified, non-overlapping visual-regime rules.
INIT_HORIZON_S = 0.5
CONTINUOUS_MAX_MISSES = 3                  # 3 / 15 Hz = 0.20 s
GAP_COAST_MIN_MISSES = 4                   # >3 missing frames
REINITIALIZE_AFTER_GAP_S = 1.5             # strictly longer than 1.5 s
CONTINUOUS_CHUNK_S = 1.0                   # manuscript protocol context only

ROTATE_TO_HORIZONTAL = True
PLOT_BORDER_M = 2.0
MAIN_BBOX_STRIDE = 2
MAIN_DOT_STRIDE = 6
CONTEXT_TRAJECTORY_STRIDE = 6
CONTEXT_HISTORY_S = 3.0
SHOW_OTHER_PEDESTRIANS = False

# Region-arrow layout: "single_row" makes the non-overlapping temporal
# segmentation visually explicit; switch to "stacked" if desired.
REGIME_ARROW_LAYOUT = "stacked"

# Optional manual replacement of the automatically generated visual spans.
# Frame intervals are inclusive. Leave None for automatic spans.
# Example:
# REGIME_OVERRIDE = {
#     "initialization": [(850, 857), (970, 977)],
#     "continuous": [(858, 920), (945, 969), (978, 1020)],
#     "gap_coasting": [(921, 944)],
# }
REGIME_OVERRIDE = None

# Main trajectory fade: beginning -> end.
MAIN_ALPHA_MIN = 0.50
MAIN_ALPHA_MAX = 1.00

# Pedestrian centre markers are expressed in metres (data coordinates).
# Current/final states use the SAME filled-dot size as historical states.
CONTEXT_DOT_RADIUS_M = 0.15
MAIN_DOT_RADIUS_M = CONTEXT_DOT_RADIUS_M
CONTEXT_ARROW_LENGTH_M = 1.05
CONTEXT_ARROW_HEAD_WIDTH_M = 0.30
CONTEXT_ARROW_HEAD_LENGTH_M = 0.36
CONTEXT_ALPHA_MIN = 0.12
CONTEXT_ALPHA_MAX = 0.65

# Focal final direction arrow: starts at the final black filled dot.
MAIN_FINAL_ARROW_LENGTH_M = 1.20
MAIN_FINAL_ARROW_HEAD_WIDTH_M = 0.32
MAIN_FINAL_ARROW_HEAD_LENGTH_M = 0.40
MAIN_FINAL_LINEWIDTH = 1.5

# Publication geometry: full two-column width.
PAGE_WIDTH_IN = 7.0
FIG1_SIZE = (PAGE_WIDTH_IN, 3.25)
SHOW_FIG1_TITLE = False
VERIFY_GLOBAL_ALIGNMENT = True
GLOBAL_ALIGNMENT_WARN_M = 0.08
PRINT_CONTEXT_DIAGNOSTICS = True

# Figure 1 colors.
GT_COLOR = "#054D6B"
DET_COLOR = "#8F0404"
MAIN_DOT_COLOR = "#054D6B"
CONTEXT_COLOR = "#879AA3"
REGIME_COLORS = {
    "initialization": "#009E73",
    "continuous": "#1478C8",
    "gap_coasting": "#D55E00",
}

# -----------------------------------------------------------------------------
# Candidate trajectory scan
# -----------------------------------------------------------------------------
CANDIDATE_MIN_DET_SUPPORT = 0.50
CANDIDATE_MIN_LONG_GAPS = 2
CANDIDATE_MIN_DURATION_S = 5.0
CANDIDATE_MOVING_SPEED_MPS = 0.40
CANDIDATE_MIN_MOVING_FRACTION = 0.95
CANDIDATE_VELOCITY_SMOOTH_WINDOW_S = 1.0
CANDIDATE_VELOCITY_POLYORDER = 3
# At least one bounded >3-frame gap must fall in this duration range.
CANDIDATE_REQUIRED_GAP_MIN_S = 0.75
CANDIDATE_REQUIRED_GAP_MAX_S = 2.0
SEQUENCES_TO_SCAN = None  # None = all stems common to local/global GT, det, odometry

# -----------------------------------------------------------------------------
# Figure 2: representative pseudo-detection visualization
# -----------------------------------------------------------------------------
PSEUDO_LEVEL = "L2"              # moderate protocol parameters; not written in titles
PSEUDO_N_FRAMES = 300
PSEUDO_FPS = 15.0
PSEUDO_SEED = 101
PSEUDO_MARKER_SIZE_PT2 = 40
PSEUDO_MARKER_ALPHA = 0.70
PSEUDO_COLORMAP = "bone_r"       # original light -> dark temporal look

PSEUDO_FINAL_BBOX_LW = 1.5
PSEUDO_FIG_SIZE = (PAGE_WIDTH_IN / 2.0, PAGE_WIDTH_IN / 2.75)
PSEUDO_SHOW_PEDESTRIAN_LABEL = True


In [ ]:

# Cell 2 — Minimal SCAI white-theme styling
# Adapted from tracker evaluation style: Helvetica/sans-serif, compact journal typography,
# no grid, 0.8-pt axes, 300-dpi export, embedded PDF fonts.
SCAI = {
    "bg": "#FFFFFF",
    "ax_bg": "#FAFBFC",
    "spine": "#D0D0D0",
    "spine_dark": "#333333",
    "text": "#111111",
    "text_dim": "#666666",
    "ees": "#1478C8",
    "ees2": "#38A4DC",
    "spr": "#F77A6C",
}


def apply_scai_white_style() -> None:
    plt.rcParams.update({
        "figure.facecolor": SCAI["bg"],
        "axes.facecolor": SCAI["ax_bg"],
        "axes.edgecolor": SCAI["spine_dark"],
        "axes.labelcolor": SCAI["text"],
        "axes.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "xtick.color": SCAI["text"],
        "ytick.color": SCAI["text"],
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 9.0,
        "axes.labelsize": 9.0,
        "xtick.labelsize": 8.0,
        "ytick.labelsize": 8.0,
        "axes.titlesize": 9.0,
        "axes.titleweight": "bold",
        "legend.fontsize": 8.0,
        "lines.linewidth": 2.0,
        "lines.solid_capstyle": "round",
        "patch.linewidth": 0.8,
        "savefig.dpi": 300,
        # "savefig.bbox": "tight",
        "savefig.facecolor": SCAI["bg"],
        "pdf.fonttype": 42,
        "svg.fonttype": "none",
    })


def despine(ax) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(SCAI["spine_dark"])
    ax.spines["bottom"].set_color(SCAI["spine_dark"])


apply_scai_white_style()


In [ ]:

# Cell 3 — TrackEval-format GT + raw detector loading and exact local matching

def _frame_key_to_int(frame_key: Any) -> int:
    text = str(frame_key).strip()
    if "." in text:
        text = text.split(".", 1)[0]
    try:
        return int(text)
    except ValueError:
        import re
        matches = re.findall(r"\d+", text)
        if not matches:
            raise ValueError(f"Cannot convert frame key {frame_key!r} to integer")
        return int(matches[-1])


def trackeval_xyzwhd_to_internal_boxes(boxes_xyzwhd: np.ndarray) -> np.ndarray:
    """JRDB TrackEval row -> [cx, cy, cz, l, w, h, rot_z]."""
    boxes = np.asarray(boxes_xyzwhd, dtype=float)
    if boxes.ndim != 2 or boxes.shape[1] != 7:
        raise ValueError(f"Expected (N,7), got {boxes.shape}")
    if len(boxes) == 0:
        return np.empty((0, 7), dtype=float)
    out = np.empty_like(boxes, dtype=float)
    out[:, 0] = boxes[:, 2]
    out[:, 1] = -boxes[:, 0]
    out[:, 2] = 0.5 * boxes[:, 4] - boxes[:, 1]
    out[:, 3] = boxes[:, 5]
    out[:, 4] = boxes[:, 3]
    out[:, 5] = boxes[:, 4]
    out[:, 6] = -boxes[:, 6]
    return out


def load_trackeval_gt_txt(path: Path) -> dict[int, dict[int, np.ndarray]]:
    """Load either local or global TrackEval GT label_02 text."""
    if not path.exists():
        raise FileNotFoundError(path)
    frames: dict[int, dict[int, np.ndarray]] = {}
    with path.open("r", encoding="utf-8") as handle:
        for line_no, raw_line in enumerate(handle, start=1):
            line = raw_line.strip()
            if not line:
                continue
            parts = line.replace(",", " ").split()
            if len(parts) < 17:
                raise ValueError(f"{path}:{line_no}: expected >=17 columns, got {len(parts)}")
            frame = int(float(parts[0]))
            gt_id = int(float(parts[1]))
            if str(parts[2]).strip().lower() != "pedestrian":
                continue
            te_box = np.asarray([float(v) for v in parts[10:17]], dtype=float).reshape(1, 7)
            box = trackeval_xyzwhd_to_internal_boxes(te_box)[0]
            frame_map = frames.setdefault(frame, {})
            if gt_id in frame_map:
                raise ValueError(f"{path}:{line_no}: duplicate GT ID {gt_id} in frame {frame}")
            frame_map[gt_id] = box
    return frames


def _pick_detection_frames(root: dict[str, Any]) -> dict[str, Any]:
    for key in ("detections", "dets", "predictions"):
        value = root.get(key)
        if isinstance(value, dict):
            return value
    if root and all(isinstance(value, list) for value in root.values()):
        return root
    raise ValueError("Detection JSON has no frame dictionary under detections/dets/predictions")


def _parse_detection_box(entry: dict[str, Any]) -> np.ndarray:
    box = entry.get("box")
    if isinstance(box, dict):
        required = ("cx", "cy", "cz", "l", "w", "h", "rot_z")
        missing = [key for key in required if key not in box]
        if missing:
            raise ValueError(f"Detection box missing {missing}: {box!r}")
        return np.asarray([float(box[key]) for key in required], dtype=float)
    if isinstance(box, (list, tuple)) and len(box) == 7:
        return np.asarray([float(v) for v in box], dtype=float)
    raise ValueError(f"Unsupported detection box: {box!r}")


def load_local_detections_json(path: Path, score_min: float | None = None) -> dict[int, np.ndarray]:
    """Load detector boxes in the original JRDB local/base frame."""
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as handle:
        root = json.load(handle)
    frame_dict = _pick_detection_frames(root)
    out: dict[int, np.ndarray] = {}
    for frame_key, entries in frame_dict.items():
        if not isinstance(entries, list):
            continue
        frame = _frame_key_to_int(frame_key)
        boxes: list[np.ndarray] = []
        for entry in entries:
            if not isinstance(entry, dict):
                continue
            label_id = entry.get("label_id")
            if label_id is not None:
                cls = str(label_id).split(":", 1)[0].strip().lower()
                if cls and cls != "pedestrian":
                    continue
            score_raw = entry.get("score")
            score = float(score_raw) if score_raw is not None else None
            if score_min is not None and (score is None or score < float(score_min)):
                continue
            boxes.append(_parse_detection_box(entry))
        out[frame] = np.asarray(boxes, dtype=float).reshape(-1, 7) if boxes else np.empty((0, 7), dtype=float)
    return out


def gated_hungarian_center_matches(
    gt_xy: np.ndarray,
    det_xy: np.ndarray,
    max_distance_m: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Exact augmented one-to-one centre-distance matching used by the detector-support cache."""
    gt_xy = np.asarray(gt_xy, dtype=float).reshape(-1, 2)
    det_xy = np.asarray(det_xy, dtype=float).reshape(-1, 2)
    n_gt, n_det = len(gt_xy), len(det_xy)
    if n_gt == 0 or n_det == 0:
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.int32), np.empty(0, dtype=float)
    gate = float(max_distance_m)
    delta = gt_xy[:, None, :] - det_xy[None, :, :]
    distances = np.linalg.norm(delta, axis=2)

    size = n_gt + n_det
    unmatched_gt_cost = float(size + 1)
    large = float(1000 * (size + 1))
    cost = np.full((size, size), large, dtype=float)
    valid = distances <= gate + 1e-12
    cost[:n_gt, :n_det][valid] = distances[valid] / gate
    cost[np.arange(n_gt), n_det + np.arange(n_gt)] = unmatched_gt_cost
    cost[n_gt + np.arange(n_det), np.arange(n_det)] = 0.0
    cost[n_gt:, n_det:] = 0.0

    rows, cols = linear_sum_assignment(cost)
    keep = (rows < n_gt) & (cols < n_det)
    rows, cols = rows[keep], cols[keep]
    if len(rows):
        keep_valid = distances[rows, cols] <= gate + 1e-12
        rows, cols = rows[keep_valid], cols[keep_valid]
    return rows.astype(np.int32), cols.astype(np.int32), distances[rows, cols].astype(float)


def build_frame_matches(
    local_gt_by_frame: dict[int, dict[int, np.ndarray]],
    detections_by_frame: dict[int, np.ndarray],
    max_distance_m: float = 0.30,
) -> dict[int, dict[int, int]]:
    """frame -> original GT ID -> raw detector index."""
    out: dict[int, dict[int, int]] = {}
    for frame in sorted(set(local_gt_by_frame) | set(detections_by_frame)):
        gt_map = local_gt_by_frame.get(frame, {})
        gt_ids = np.asarray(sorted(gt_map), dtype=np.int64)
        gt_boxes = (
            np.asarray([gt_map[int(gid)] for gid in gt_ids], dtype=float).reshape(-1, 7)
            if len(gt_ids) else np.empty((0, 7), dtype=float)
        )
        det_boxes = np.asarray(detections_by_frame.get(frame, np.empty((0, 7))), dtype=float).reshape(-1, 7)
        rows, cols, _ = gated_hungarian_center_matches(gt_boxes[:, :2], det_boxes[:, :2], max_distance_m)
        out[frame] = {int(gt_ids[r]): int(c) for r, c in zip(rows.tolist(), cols.tolist())}
    return out


In [ ]:

# Cell 4 — SteamLO odometry: exact tracker_eval local -> global convention
@dataclass(frozen=True)
class Pose:
    t: np.ndarray  # (3,)
    q: np.ndarray  # qx,qy,qz,qw


def _quat_to_R(qx: float, qy: float, qz: float, qw: float) -> np.ndarray:
    n = math.sqrt(qx*qx + qy*qy + qz*qz + qw*qw)
    if n < 1e-12:
        return np.eye(3, dtype=np.float64)
    qx, qy, qz, qw = qx/n, qy/n, qz/n, qw/n
    xx, yy, zz = qx*qx, qy*qy, qz*qz
    xy, xz, yz = qx*qy, qx*qz, qy*qz
    wx, wy, wz = qw*qx, qw*qy, qw*qz
    return np.asarray([
        [1.0 - 2.0*(yy + zz), 2.0*(xy - wz),       2.0*(xz + wy)],
        [2.0*(xy + wz),       1.0 - 2.0*(xx + zz), 2.0*(yz - wx)],
        [2.0*(xz - wy),       2.0*(yz + wx),       1.0 - 2.0*(xx + yy)],
    ], dtype=np.float64)


def _quat_yaw(qx: float, qy: float, qz: float, qw: float) -> float:
    n = math.sqrt(qx*qx + qy*qy + qz*qz + qw*qw)
    if n < 1e-12:
        return 0.0
    qx, qy, qz, qw = qx/n, qy/n, qz/n, qw/n
    return float(math.atan2(2.0*(qw*qz + qx*qy), 1.0 - 2.0*(qy*qy + qz*qz)))


def load_odometry_csv(path: Path) -> dict[int, Pose]:
    if not path.exists():
        raise FileNotFoundError(path)
    poses: dict[int, Pose] = {}
    with path.open("r", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        required = {"timestamp_ns", "x", "y", "z", "qx", "qy", "qz", "qw"}
        if not required.issubset(set(reader.fieldnames or [])):
            raise ValueError(f"Odometry CSV missing required columns; have {reader.fieldnames}")
        for i, row in enumerate(reader):
            poses[i] = Pose(
                t=np.asarray([float(row["x"]), float(row["y"]), float(row["z"])], dtype=np.float64),
                q=np.asarray([float(row["qx"]), float(row["qy"]), float(row["qz"]), float(row["qw"])], dtype=np.float64),
            )
    if not poses:
        raise ValueError(f"Odometry CSV is empty: {path}")
    return poses


def _world_from_sensor_terms(pose: Pose) -> tuple[np.ndarray, np.ndarray, float]:
    qx, qy, qz, qw = map(float, pose.q)
    R = _quat_to_R(qx, qy, qz, qw)
    t = pose.t.astype(np.float64, copy=False)
    yaw = _quat_yaw(qx, qy, qz, qw)
    return R, t, float(yaw)


def transform_box7_local_to_global(
    box7: np.ndarray,
    pose: Pose,
) -> np.ndarray:
    """Exact tracker_eval convention: p_g = R p_l + t; yaw_g = yaw_l - ego_yaw."""
    b = np.asarray(box7, dtype=np.float64).reshape(7)
    R, t, yaw = _world_from_sensor_terms(pose)
    out = b.copy()
    out[:3] = R @ b[:3] + t
    out[6] = float(b[6] - yaw)
    return out


def transform_detection_dict_to_global(
    detections_local: dict[int, np.ndarray],
    poses: dict[int, Pose],
) -> dict[int, np.ndarray]:
    out: dict[int, np.ndarray] = {}
    for frame, boxes in detections_local.items():
        boxes = np.asarray(boxes, dtype=float).reshape(-1, 7)
        pose = poses.get(int(frame))
        if pose is None:
            raise KeyError(f"No SteamLO pose for frame {frame}")
        out[frame] = np.asarray([
            transform_box7_local_to_global(box, pose)
            for box in boxes
        ], dtype=float).reshape(-1, 7) if len(boxes) else np.empty((0, 7), dtype=float)
    return out


In [ ]:
# Cell 5 — Reusable caches + candidate trajectory scan

def _stable_signature(payload: dict[str, Any]) -> str:
    text = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


def _file_state(path: Path) -> dict[str, Any]:
    path = Path(path)
    stat = path.stat()
    return {
        "path": str(path.resolve()),
        "size": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
    }


def _safe_filename(value: str) -> str:
    return str(value).replace("/", "__").replace("\\", "__").replace(":", "_")


def _frame_match_cache_path(sequence: str) -> Path:
    local_path = JRDB_LOCAL_GT_FOLDER / f"{sequence}.txt"
    det_path = JRDB_DETECTIONS_DIR / f"{sequence}.json"
    signature = _stable_signature({
        "version": 1,
        "sequence": sequence,
        "local_gt": _file_state(local_path),
        "detections": _file_state(det_path),
        "det_match_max_dist_m": float(DET_MATCH_MAX_DIST_M),
        "det_score_min": DET_SCORE_MIN,
    })
    folder = CACHE_DIR / "frame_matches"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"{_safe_filename(sequence)}__{signature}.json"


def get_frame_matches_cached(
    sequence: str,
    local_gt: dict[int, dict[int, np.ndarray]],
    detections: dict[int, np.ndarray],
    *,
    overwrite: bool | None = None,
) -> dict[int, dict[int, int]]:
    overwrite = OVERWRITE_CACHE if overwrite is None else bool(overwrite)
    cache_path = _frame_match_cache_path(sequence)
    if cache_path.exists() and not overwrite:
        raw = json.loads(cache_path.read_text(encoding="utf-8"))
        return {
            int(frame): {int(gid): int(det_idx) for gid, det_idx in mapping.items()}
            for frame, mapping in raw.items()
        }

    matches = build_frame_matches(
        local_gt,
        detections,
        max_distance_m=DET_MATCH_MAX_DIST_M,
    )
    cache_path.write_text(
        json.dumps(
            {
                str(frame): {str(gid): int(det_idx) for gid, det_idx in mapping.items()}
                for frame, mapping in matches.items()
            },
            sort_keys=True,
        ),
        encoding="utf-8",
    )
    return matches


def _contiguous_runs(frames: np.ndarray) -> list[np.ndarray]:
    frames = np.asarray(frames, dtype=int)
    if len(frames) == 0:
        return []
    cuts = np.where(np.diff(frames) != 1)[0] + 1
    return [run for run in np.split(np.arange(len(frames)), cuts) if len(run)]


def smoothed_speed_mps(
    frames: np.ndarray,
    xy: np.ndarray,
    fps: float,
    window_s: float,
    polyorder: int,
) -> np.ndarray:
    """Savitzky-Golay XY smoothing + time derivative on GT-contiguous runs."""
    frames = np.asarray(frames, dtype=int)
    xy = np.asarray(xy, dtype=float).reshape(-1, 2)
    speeds = np.full(len(frames), np.nan, dtype=float)
    target_window = max(3, int(round(float(window_s) * float(fps))))
    if target_window % 2 == 0:
        target_window += 1

    for idx in _contiguous_runs(frames):
        if len(idx) < 2:
            continue
        sub_xy = xy[idx]
        sub_frames = frames[idx]
        max_odd = len(idx) if len(idx) % 2 == 1 else len(idx) - 1
        window = min(target_window, max_odd)
        if window >= max(polyorder + 2, 3):
            if window % 2 == 0:
                window -= 1
            smooth = np.column_stack([
                savgol_filter(
                    sub_xy[:, d],
                    window_length=window,
                    polyorder=min(polyorder, window - 2),
                    mode="interp",
                )
                for d in range(2)
            ])
        else:
            smooth = sub_xy
        t = sub_frames.astype(float) / float(fps)
        vel = np.column_stack([np.gradient(smooth[:, d], t) for d in range(2)])
        speeds[idx] = np.linalg.norm(vel, axis=1)
    return speeds


def long_detector_gap_spans(
    gt_frames: list[int],
    supported_frames: list[int],
    min_missing_frames: int = 4,
) -> list[tuple[int, int, int]]:
    """Return (pre-supported frame, post-supported frame, missing-frame count)."""
    gt_set = set(map(int, gt_frames))
    supported_frames = sorted(map(int, supported_frames))
    spans: list[tuple[int, int, int]] = []
    for prev, cur in zip(supported_frames[:-1], supported_frames[1:]):
        missing = int(cur) - int(prev) - 1
        if missing < int(min_missing_frames):
            continue
        if all(frame in gt_set for frame in range(int(prev) + 1, int(cur))):
            spans.append((int(prev), int(cur), int(missing)))
    return spans


def discover_available_sequences() -> list[str]:
    local = {p.stem for p in JRDB_LOCAL_GT_FOLDER.glob("*.txt")}
    global_ = {p.stem for p in JRDB_GLOBAL_GT_FOLDER.glob("*.txt")}
    det = {p.stem for p in JRDB_DETECTIONS_DIR.glob("*.json")}
    odom = {p.stem for p in JRDB_ODOMETRY_DIR.glob("*.csv")}
    common = sorted(local & global_ & det & odom)
    if not common:
        raise RuntimeError(
            "No common sequence stems found across local GT, global GT, detections, and odometry.\n"
            f"local={JRDB_LOCAL_GT_FOLDER}\n"
            f"global={JRDB_GLOBAL_GT_FOLDER}\n"
            f"det={JRDB_DETECTIONS_DIR}\n"
            f"odom={JRDB_ODOMETRY_DIR}"
        )
    return common


def _candidate_cache_path(sequences: list[str]) -> Path:
    source_state = {}
    for sequence in sequences:
        source_state[sequence] = {
            "local_gt": _file_state(JRDB_LOCAL_GT_FOLDER / f"{sequence}.txt"),
            "global_gt": _file_state(JRDB_GLOBAL_GT_FOLDER / f"{sequence}.txt"),
            "detections": _file_state(JRDB_DETECTIONS_DIR / f"{sequence}.json"),
            "odometry": _file_state(JRDB_ODOMETRY_DIR / f"{sequence}.csv"),
        }
    signature = _stable_signature({
        "version": 2,
        "sources": source_state,
        "fps": float(FPS),
        "det_match_max_dist_m": float(DET_MATCH_MAX_DIST_M),
        "det_score_min": DET_SCORE_MIN,
        "min_det_support": float(CANDIDATE_MIN_DET_SUPPORT),
        "min_long_gaps": int(CANDIDATE_MIN_LONG_GAPS),
        "min_duration_s": float(CANDIDATE_MIN_DURATION_S),
        "moving_speed_mps": float(CANDIDATE_MOVING_SPEED_MPS),
        "min_moving_fraction": float(CANDIDATE_MIN_MOVING_FRACTION),
        "velocity_window_s": float(CANDIDATE_VELOCITY_SMOOTH_WINDOW_S),
        "velocity_polyorder": int(CANDIDATE_VELOCITY_POLYORDER),
        "gap_min_missing_frames": int(GAP_COAST_MIN_MISSES),
        "required_gap_min_s": float(CANDIDATE_REQUIRED_GAP_MIN_S),
        "required_gap_max_s": float(CANDIDATE_REQUIRED_GAP_MAX_S),
    })
    folder = CACHE_DIR / "candidate_scans"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"candidate_trajectories__{signature}.csv"


def scan_candidate_trajectories(
    sequences: list[str] | None = None,
    *,
    overwrite: bool | None = None,
) -> pd.DataFrame:
    sequences = discover_available_sequences() if sequences is None else list(sequences)
    overwrite = OVERWRITE_CACHE if overwrite is None else bool(overwrite)
    cache_path = _candidate_cache_path(sequences)

    if cache_path.exists() and not overwrite:
        print(f"Loading cached candidate scan: {cache_path}")
        return pd.read_csv(cache_path)

    rows: list[dict[str, Any]] = []
    for seq_i, sequence in enumerate(sequences, start=1):
        print(f"[{seq_i:02d}/{len(sequences):02d}] {sequence}")
        local_gt = load_trackeval_gt_txt(JRDB_LOCAL_GT_FOLDER / f"{sequence}.txt")
        global_gt = load_trackeval_gt_txt(JRDB_GLOBAL_GT_FOLDER / f"{sequence}.txt")
        detections = load_local_detections_json(
            JRDB_DETECTIONS_DIR / f"{sequence}.json",
            score_min=DET_SCORE_MIN,
        )
        matches = get_frame_matches_cached(sequence, local_gt, detections, overwrite=overwrite)

        gt_ids = sorted({gid for frame_map in local_gt.values() for gid in frame_map})
        for gid in gt_ids:
            gt_frames = sorted(frame for frame, frame_map in local_gt.items() if gid in frame_map)
            global_frames = sorted(frame for frame, frame_map in global_gt.items() if gid in frame_map)
            common_frames = sorted(set(gt_frames) & set(global_frames))
            if len(common_frames) < 2:
                continue

            duration_s = (common_frames[-1] - common_frames[0] + 1) / float(FPS)
            if duration_s < CANDIDATE_MIN_DURATION_S:
                continue

            supported = [frame for frame in gt_frames if gid in matches.get(frame, {})]
            support_fraction = len(supported) / len(gt_frames) if gt_frames else np.nan
            if support_fraction < CANDIDATE_MIN_DET_SUPPORT:
                continue

            gaps = long_detector_gap_spans(
                gt_frames,
                supported,
                min_missing_frames=GAP_COAST_MIN_MISSES,
            )
            if len(gaps) < CANDIDATE_MIN_LONG_GAPS:
                continue

            gap_durations_s = np.asarray([missing / float(FPS) for _, _, missing in gaps], dtype=float)
            target_gap_mask = (
                (gap_durations_s >= float(CANDIDATE_REQUIRED_GAP_MIN_S) - 1e-12)
                & (gap_durations_s <= float(CANDIDATE_REQUIRED_GAP_MAX_S) + 1e-12)
            )
            if not np.any(target_gap_mask):
                continue

            xy = np.asarray([global_gt[frame][gid][:2] for frame in common_frames], dtype=float)
            speeds = smoothed_speed_mps(
                np.asarray(common_frames),
                xy,
                fps=FPS,
                window_s=CANDIDATE_VELOCITY_SMOOTH_WINDOW_S,
                polyorder=CANDIDATE_VELOCITY_POLYORDER,
            )
            valid_speed = np.isfinite(speeds)
            if not np.any(valid_speed):
                continue
            moving_fraction = float(np.mean(speeds[valid_speed] > CANDIDATE_MOVING_SPEED_MPS))
            if moving_fraction < CANDIDATE_MIN_MOVING_FRACTION:
                continue

            rows.append({
                "sequence": sequence,
                "gt_id": int(gid),
                "first_frame": int(common_frames[0]),
                "last_frame": int(common_frames[-1]),
                "duration_s": float(duration_s),
                "n_gt_frames": int(len(gt_frames)),
                "det_support_fraction": float(support_fraction),
                "n_long_gaps": int(len(gaps)),
                "n_gaps_0p75_to_2s": int(np.sum(target_gap_mask)),
                "max_long_gap_frames": int(max(g[2] for g in gaps)),
                "max_long_gap_s": float(np.max(gap_durations_s)),
                "target_gap_durations_s": ", ".join(
                    f"{value:.2f}" for value in gap_durations_s[target_gap_mask]
                ),
                "moving_fraction": float(moving_fraction),
                "median_smoothed_speed_mps": float(np.nanmedian(speeds)),
            })

    columns = [
        "sequence", "gt_id", "first_frame", "last_frame", "duration_s",
        "n_gt_frames", "det_support_fraction", "n_long_gaps",
        "n_gaps_0p75_to_2s", "max_long_gap_frames", "max_long_gap_s",
        "target_gap_durations_s", "moving_fraction", "median_smoothed_speed_mps",
    ]
    result = pd.DataFrame(rows, columns=columns)
    if not result.empty:
        result = result.sort_values(
            ["n_long_gaps", "n_gaps_0p75_to_2s", "duration_s", "det_support_fraction"],
            ascending=[False, False, False, False],
        ).reset_index(drop=True)

    result.to_csv(cache_path, index=False)
    print(f"Saved candidate scan cache: {cache_path}")
    return result


In [ ]:
# Cell 6 — Run candidate scan and display ALL matching sequence / GT IDs
scan_sequences = discover_available_sequences() if SEQUENCES_TO_SCAN is None else list(SEQUENCES_TO_SCAN)
candidates = scan_candidate_trajectories(
    scan_sequences,
    overwrite=OVERWRITE_CACHE,
)

print("\nCandidate trajectories satisfying all configured criteria:")
if candidates.empty:
    print("  None found. Adjust the thresholds in Cell 1 if needed.")
else:
    # Explicitly disable Pandas row/column truncation for this table.
    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.width", 240,
        "display.max_colwidth", 120,
    ):
        display(candidates)

print(f"\nCandidate cache directory: {CACHE_DIR / 'candidate_scans'}")


In [ ]:
# Cell 7 — Figure 1 preparation + plotting helpers

def split_detector_intervals(
    supported_frames: list[int],
    gt_frames: set[int],
    max_missing_frames: int,
) -> list[list[int]]:
    """Evaluation-style supported intervals joined only across short GT-present gaps."""
    if not supported_frames:
        return []
    supported_frames = sorted(map(int, supported_frames))
    intervals: list[list[int]] = [[supported_frames[0]]]
    for value in supported_frames[1:]:
        current = int(value)
        previous = int(intervals[-1][-1])
        missing = current - previous - 1
        gt_continuous = all(frame in gt_frames for frame in range(previous + 1, current))
        if 0 <= missing <= int(max_missing_frames) and gt_continuous:
            intervals[-1].append(current)
        else:
            intervals.append([current])
    return intervals


def alpha_from_frame(frame: int, start: int, end: int, alpha_min: float, alpha_max: float) -> float:
    if end <= start:
        return float(alpha_max)
    frac = np.clip((float(frame) - start) / (end - start), 0.0, 1.0)
    return float(alpha_min + frac * (alpha_max - alpha_min))


def rotation_angle_from_boxes(boxes: list[np.ndarray]) -> float:
    centres = np.asarray([box[:2] for box in boxes], dtype=float)
    if len(centres) < 2:
        return 0.0
    centered = centres - centres.mean(axis=0, keepdims=True)
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    direction = vh[0]
    if np.dot(direction, centres[-1] - centres[0]) < 0:
        direction = -direction
    return -math.atan2(float(direction[1]), float(direction[0]))


def rotate_points(points_xy: np.ndarray, angle_rad: float, origin_xy: np.ndarray) -> np.ndarray:
    points = np.asarray(points_xy, dtype=float).reshape(-1, 2)
    origin = np.asarray(origin_xy, dtype=float).reshape(2)
    c, s = math.cos(angle_rad), math.sin(angle_rad)
    R = np.asarray([[c, -s], [s, c]], dtype=float)
    return (points - origin[None, :]) @ R.T + origin[None, :]


def rotate_box(box: np.ndarray, angle_rad: float, origin_xy: np.ndarray) -> np.ndarray:
    out = np.asarray(box, dtype=float).reshape(7).copy()
    out[:2] = rotate_points(out[:2].reshape(1, 2), angle_rad, origin_xy)[0]
    out[6] = float(out[6] + angle_rad)
    return out


def _bev_corners_ccw(box: np.ndarray) -> np.ndarray:
    cx, cy, _, length, width, _, yaw = map(float, box)
    c, s = math.cos(yaw), math.sin(yaw)
    u = np.asarray([c, s])
    v = np.asarray([-s, c])
    hu, hv = 0.5 * length * u, 0.5 * width * v
    centre = np.asarray([cx, cy])
    return np.asarray([
        centre + hu + hv,
        centre - hu + hv,
        centre - hu - hv,
        centre + hu - hv,
    ])


def draw_box_outline(
    ax,
    box: np.ndarray,
    color: str,
    alpha: float,
    linewidth: float,
    zorder: int,
) -> None:
    """Draw only the rectangle outline — no centre/front orientation line."""
    corners = _bev_corners_ccw(box)
    ax.add_patch(Polygon(
        corners,
        closed=True,
        fill=False,
        edgecolor=color,
        linewidth=linewidth,
        alpha=alpha,
        zorder=zorder,
    ))


def _motion_unit_direction(
    frames: list[int],
    boxes: list[np.ndarray],
    fallback_yaw: float,
    lookback_frames: int = 8,
) -> np.ndarray:
    if len(boxes) >= 2:
        use = min(len(boxes), max(2, int(lookback_frames)))
        delta = np.asarray(boxes[-1][:2]) - np.asarray(boxes[-use][:2])
        norm = np.linalg.norm(delta)
        if norm > 1e-6:
            return delta / norm
    return np.asarray([math.cos(fallback_yaw), math.sin(fallback_yaw)], dtype=float)


def _in_bounds(xy: np.ndarray, xlim: tuple[float, float], ylim: tuple[float, float]) -> bool:
    x, y = map(float, xy)
    return xlim[0] <= x <= xlim[1] and ylim[0] <= y <= ylim[1]


def _build_visual_regimes(
    gt_frames: list[int],
    supported_frames: list[int],
) -> dict[str, list[tuple[int, int]]]:
    """Build a simplified, mutually non-overlapping explanatory segmentation.

    Initialize:
      - first detector-supported frame, for ceil(0.5 s * FPS) frames;
      - again at the first supported frame after a bounded gap >1.5 s.
    Continuous:
      - within intervals joined across <=3 consecutive misses;
      - starts after Initialize when that interval has an initialization phase;
      - otherwise starts immediately at the first supported post-gap frame.
    Gap coasting:
      - missing frames only, for every bounded gap >3 misses.
    """
    gt_frames = sorted(map(int, gt_frames))
    supported_frames = sorted(map(int, supported_frames))
    if not gt_frames or not supported_frames:
        return {"initialization": [], "continuous": [], "gap_coasting": []}

    gt_set = set(gt_frames)
    init_frames = int(math.ceil(float(INIT_HORIZON_S) * float(FPS) - 1e-12))

    # Missing-frame-only bounded gaps >3.
    gaps_raw = long_detector_gap_spans(
        gt_frames,
        supported_frames,
        min_missing_frames=GAP_COAST_MIN_MISSES,
    )
    gap_coasting = [
        (int(pre + 1), int(post - 1))
        for pre, post, missing in gaps_raw
        if missing >= GAP_COAST_MIN_MISSES and post - pre > 1
    ]
    gap_by_post = {int(post): (int(pre), int(missing)) for pre, post, missing in gaps_raw}

    # Same interval primitive as the continuity analysis: >3 misses split it.
    intervals = split_detector_intervals(
        supported_frames,
        gt_set,
        max_missing_frames=CONTINUOUS_MAX_MISSES,
    )

    initialization: list[tuple[int, int]] = []
    continuous: list[tuple[int, int]] = []

    for interval_index, interval in enumerate(intervals):
        if not interval:
            continue
        segment_start = int(interval[0])
        segment_end = int(interval[-1])

        needs_init = interval_index == 0
        if not needs_init:
            preceding = gap_by_post.get(segment_start)
            if preceding is not None:
                _, missing = preceding
                gap_s = float(missing) / float(FPS)
                needs_init = gap_s > float(REINITIALIZE_AFTER_GAP_S) + 1e-12
            else:
                # A split not represented as a bounded detector gap implies GT
                # discontinuity / new visible GT segment: treat as a fresh start.
                previous_supported = max((f for f in supported_frames if f < segment_start), default=None)
                if previous_supported is None:
                    needs_init = True
                else:
                    needs_init = not all(
                        frame in gt_set
                        for frame in range(int(previous_supported) + 1, segment_start)
                    )

        if needs_init:
            init_start = segment_start
            # Clip to this short-gap interval so initialization never crosses a
            # >3-frame gap and can never overlap gap coasting.
            init_end = min(segment_end, init_start + init_frames - 1)
            initialization.append((init_start, init_end))
            continuous_start = init_end + 1
        else:
            continuous_start = segment_start

        if continuous_start <= segment_end:
            continuous.append((int(continuous_start), int(segment_end)))

    # Defensive audit: the three displayed populations must not overlap by frame.
    occupied: dict[int, str] = {}
    for key, spans in (
        ("gap_coasting", gap_coasting),
        ("initialization", initialization),
        ("continuous", continuous),
    ):
        for start, end in spans:
            for frame in range(int(start), int(end) + 1):
                previous = occupied.get(frame)
                if previous is not None:
                    raise RuntimeError(
                        f"Visual regime overlap at frame {frame}: {previous} and {key}"
                    )
                occupied[frame] = key

    return {
        "initialization": initialization,
        "continuous": continuous,
        "gap_coasting": gap_coasting,
    }


@dataclass
class Figure1Payload:
    main_frames: list[int]
    main_gt: dict[int, np.ndarray]
    main_det_global: dict[int, np.ndarray | None]
    context_tracks: dict[int, list[tuple[int, np.ndarray]]]
    xlim: tuple[float, float]
    scene_ylim: tuple[float, float]
    regimes: dict[str, list[tuple[int, int]]]
    cosmetic_angle_rad: float
    alignment_errors_m: np.ndarray
    context_window: tuple[int, int]
    context_total_tracks: int
    context_visible_tracks: int
    context_latest_visible_tracks: int
    context_min_same_frame_distance_m: float
    context_final_frame_nearest_distance_m: float


def prepare_figure1_payload(
    sequence: str,
    selected_gt_id: int,
    frame_start: int | None = None,
    frame_end: int | None = None,
) -> Figure1Payload:
    local_gt = load_trackeval_gt_txt(JRDB_LOCAL_GT_FOLDER / f"{sequence}.txt")
    global_gt = load_trackeval_gt_txt(JRDB_GLOBAL_GT_FOLDER / f"{sequence}.txt")
    detections_local = load_local_detections_json(
        JRDB_DETECTIONS_DIR / f"{sequence}.json",
        score_min=DET_SCORE_MIN,
    )
    poses = load_odometry_csv(JRDB_ODOMETRY_DIR / f"{sequence}.csv")
    matches = get_frame_matches_cached(
        sequence,
        local_gt,
        detections_local,
        overwrite=OVERWRITE_CACHE,
    )

    gt_frames_all = sorted(frame for frame, fm in global_gt.items() if selected_gt_id in fm)
    if not gt_frames_all:
        raise KeyError(f"GT ID {selected_gt_id} not found in global GT for {sequence}")
    lo = gt_frames_all[0] if frame_start is None else max(int(frame_start), gt_frames_all[0])
    hi = gt_frames_all[-1] if frame_end is None else min(int(frame_end), gt_frames_all[-1])
    main_frames = [frame for frame in gt_frames_all if lo <= frame <= hi]
    if not main_frames:
        raise ValueError("Selected frame range contains no focal GT frames")

    main_gt_global = {
        frame: np.asarray(global_gt[frame][selected_gt_id], dtype=float).copy()
        for frame in main_frames
    }
    main_det_global: dict[int, np.ndarray | None] = {}
    alignment_errors: list[float] = []

    # Match in local coordinates; transform only the selected matched detection.
    for frame in main_frames:
        det_idx = matches.get(frame, {}).get(int(selected_gt_id))
        det_boxes = np.asarray(
            detections_local.get(frame, np.empty((0, 7))),
            dtype=float,
        ).reshape(-1, 7)
        if det_idx is None or not (0 <= int(det_idx) < len(det_boxes)):
            main_det_global[frame] = None
        else:
            pose = poses.get(frame)
            if pose is None:
                raise KeyError(f"No SteamLO pose for frame {frame}")
            main_det_global[frame] = transform_box7_local_to_global(
                det_boxes[int(det_idx)],
                pose,
            )

        # Sanity check: independently transform local GT and compare to gt__global.
        local_box = local_gt.get(frame, {}).get(int(selected_gt_id))
        pose = poses.get(frame)
        if local_box is not None and pose is not None:
            transformed_gt = transform_box7_local_to_global(
                local_box,
                pose,
            )
            alignment_errors.append(
                float(np.linalg.norm(transformed_gt[:2] - main_gt_global[frame][:2]))
            )

    # One fixed plot-only rotation for every object.
    cosmetic_angle = (
        rotation_angle_from_boxes([main_gt_global[f] for f in main_frames])
        if ROTATE_TO_HORIZONTAL
        else 0.0
    )
    origin = np.mean(
        np.asarray([main_gt_global[f][:2] for f in main_frames]),
        axis=0,
    )
    main_gt = {
        f: rotate_box(b, cosmetic_angle, origin)
        for f, b in main_gt_global.items()
    }
    main_det_global = {
        f: (None if b is None else rotate_box(b, cosmetic_angle, origin))
        for f, b in main_det_global.items()
    }

    centres = np.asarray([main_gt[f][:2] for f in main_frames], dtype=float)
    xlim = (
        float(centres[:, 0].min() - PLOT_BORDER_M),
        float(centres[:, 0].max() + PLOT_BORDER_M),
    )
    scene_ylim = (
        float(centres[:, 1].min() - PLOT_BORDER_M),
        float(centres[:, 1].max() + PLOT_BORDER_M),
    )

    # Context rule: only the last CONTEXT_HISTORY_S of the selected plotting
    # interval (or the entire selected interval if shorter). Keep ALL samples in
    # this temporal window; clip spatially only when drawing. This avoids an
    # artifact where a pedestrian that later leaves the view gets an arrow at its
    # last in-bounds historical point.
    context_start = max(lo, hi - int(math.ceil(CONTEXT_HISTORY_S * FPS)) + 1)
    context_tracks: dict[int, list[tuple[int, np.ndarray]]] = {}
    same_frame_nearest: list[float] = []
    final_nearest = float("nan")

    if SHOW_OTHER_PEDESTRIANS:
        for frame in range(context_start, hi + 1):
            frame_map = global_gt.get(frame, {})
            main_box_raw = frame_map.get(int(selected_gt_id))
            other_centres_raw: list[np.ndarray] = []
            for gid, box in frame_map.items():
                if int(gid) == int(selected_gt_id):
                    continue
                box_arr = np.asarray(box, dtype=float)
                other_centres_raw.append(box_arr[:2])
                rotated = rotate_box(box_arr, cosmetic_angle, origin)
                context_tracks.setdefault(int(gid), []).append((int(frame), rotated))

            if main_box_raw is not None and other_centres_raw:
                other_xy = np.asarray(other_centres_raw, dtype=float)
                d = np.linalg.norm(other_xy - np.asarray(main_box_raw[:2], dtype=float), axis=1)
                same_frame_nearest.append(float(np.min(d)))
                if frame == hi:
                    final_nearest = float(np.min(d))

    context_total_tracks = len(context_tracks)
    context_visible_tracks = 0
    context_latest_visible_tracks = 0
    for track in context_tracks.values():
        if any(_in_bounds(box[:2], xlim, scene_ylim) for _, box in track):
            context_visible_tracks += 1
        if track and _in_bounds(track[-1][1][:2], xlim, scene_ylim):
            context_latest_visible_tracks += 1

    # Detector support/regimes come from local-frame matching.
    local_gt_frames = [
        frame for frame in main_frames
        if selected_gt_id in local_gt.get(frame, {})
    ]
    supported = [
        frame for frame in local_gt_frames
        if selected_gt_id in matches.get(frame, {})
    ]

    if REGIME_OVERRIDE is None:
        regimes = _build_visual_regimes(local_gt_frames, supported)
    else:
        regimes = {
            "initialization": [
                (int(a), int(b)) for a, b in REGIME_OVERRIDE.get("initialization", [])
            ],
            "continuous": [
                (int(a), int(b)) for a, b in REGIME_OVERRIDE.get("continuous", [])
            ],
            "gap_coasting": [
                (int(a), int(b))
                for a, b in REGIME_OVERRIDE.get(
                    "gap_coasting",
                    REGIME_OVERRIDE.get("long_gaps", []),
                )
            ],
        }

    return Figure1Payload(
        main_frames=main_frames,
        main_gt=main_gt,
        main_det_global=main_det_global,
        context_tracks=context_tracks,
        xlim=xlim,
        scene_ylim=scene_ylim,
        regimes=regimes,
        cosmetic_angle_rad=cosmetic_angle,
        alignment_errors_m=np.asarray(alignment_errors, dtype=float),
        context_window=(int(context_start), int(hi)),
        context_total_tracks=int(context_total_tracks),
        context_visible_tracks=int(context_visible_tracks),
        context_latest_visible_tracks=int(context_latest_visible_tracks),
        context_min_same_frame_distance_m=(
            float(np.min(same_frame_nearest)) if same_frame_nearest else float("nan")
        ),
        context_final_frame_nearest_distance_m=float(final_nearest),
    )


def _draw_context_track(
    ax,
    track: list[tuple[int, np.ndarray]],
    main_start: int,
    main_end: int,
    xlim: tuple[float, float],
    ylim: tuple[float, float],
) -> None:
    track = sorted(track, key=lambda x: x[0])
    if not track:
        return

    sampled = track[::max(1, int(CONTEXT_TRAJECTORY_STRIDE))]
    if sampled[-1][0] != track[-1][0]:
        sampled.append(track[-1])

    # Every visible state is the same filled-dot size, including the latest.
    for frame, box in sampled:
        if not _in_bounds(box[:2], xlim, ylim):
            continue
        alpha = alpha_from_frame(
            frame,
            main_start,
            main_end,
            CONTEXT_ALPHA_MIN,
            CONTEXT_ALPHA_MAX,
        )
        ax.add_patch(Circle(
            tuple(box[:2]),
            radius=CONTEXT_DOT_RADIUS_M,
            facecolor=CONTEXT_COLOR,
            edgecolor="none",
            alpha=alpha,
            zorder=1.5,
        ))

    # Arrow only at the pedestrian's ACTUAL latest state in the temporal window,
    # and only if that latest state itself is inside the plotted region.
    last_frame, last_box = track[-1]
    if not _in_bounds(last_box[:2], xlim, ylim):
        return
    alpha = alpha_from_frame(
        last_frame,
        main_start,
        main_end,
        CONTEXT_ALPHA_MIN,
        CONTEXT_ALPHA_MAX,
    )
    direction = _motion_unit_direction(
        [f for f, _ in track],
        [b for _, b in track],
        fallback_yaw=float(last_box[6]),
        lookback_frames=max(2, int(round(0.5 * FPS))),
    )
    ax.arrow(
        float(last_box[0]),
        float(last_box[1]),
        float(direction[0] * CONTEXT_ARROW_LENGTH_M),
        float(direction[1] * CONTEXT_ARROW_LENGTH_M),
        width=0.014,
        head_width=CONTEXT_ARROW_HEAD_WIDTH_M,
        head_length=CONTEXT_ARROW_HEAD_LENGTH_M,
        length_includes_head=True,
        fc=CONTEXT_COLOR,
        ec=CONTEXT_COLOR,
        alpha=alpha,
        linewidth=0.7,
        zorder=2.5,
    )


def _draw_regime_arrow(ax, x0: float, x1: float, y: float, color: str) -> None:
    # Preserve temporal direction; the plot-only rotation usually makes it L->R.
    if abs(float(x1) - float(x0)) < 0.10:
        x1 = float(x0) + (0.10 if x1 >= x0 else -0.10)
    ax.add_patch(FancyArrowPatch(
        (float(x0), float(y)),
        (float(x1), float(y)),
        arrowstyle="-|>",
        mutation_scale=16,
        linewidth=2.0,
        color=color,
        shrinkA=0,
        shrinkB=0,
        zorder=12,
    ))


def plot_figure1(payload: Figure1Payload, sequence: str, gt_id: int) -> plt.Figure:
    apply_scai_white_style()
    fig, ax = plt.subplots(figsize=FIG1_SIZE, dpi=300)
    main_start, main_end = payload.main_frames[0], payload.main_frames[-1]

    # Optional surrounding pedestrians.
    if SHOW_OTHER_PEDESTRIANS:
        for track in payload.context_tracks.values():
            _draw_context_track(
                ax,
                track,
                main_start,
                main_end,
                payload.xlim,
                payload.scene_ylim,
            )

    # Sparse focal GT + detector boxes. No orientation line inside rectangles.
    bbox_frames = payload.main_frames[::max(1, int(MAIN_BBOX_STRIDE))]
    if payload.main_frames[-1] not in bbox_frames:
        bbox_frames.append(payload.main_frames[-1])

    for frame in bbox_frames:
        alpha = alpha_from_frame(
            frame,
            main_start,
            main_end,
            MAIN_ALPHA_MIN,
            MAIN_ALPHA_MAX,
        )
        draw_box_outline(
            ax,
            payload.main_gt[frame],
            GT_COLOR,
            alpha=alpha,
            linewidth=0.95,
            zorder=5,
        )
        det = payload.main_det_global.get(frame)
        if det is not None:
            draw_box_outline(
                ax,
                det,
                DET_COLOR,
                alpha=alpha,
                linewidth=1.05,
                zorder=6,
            )

    # Focal pedestrian centre dots.
    # Draw BEFORE/behind the GT and detector box outlines.
    dot_frames = payload.main_frames[::max(1, int(MAIN_DOT_STRIDE))]
    if payload.main_frames[-1] not in dot_frames:
        dot_frames.append(payload.main_frames[-1])

    for frame in dot_frames:
        alpha = alpha_from_frame(
            frame,
            main_start,
            main_end,
            MAIN_ALPHA_MIN,
            MAIN_ALPHA_MAX,
        )
        box = payload.main_gt[frame]

        ax.add_patch(
            Circle(
                tuple(box[:2]),
                radius=MAIN_DOT_RADIUS_M,
                facecolor=MAIN_DOT_COLOR,
                edgecolor="none",
                alpha=alpha,
                zorder=4,
            )
        )

    # Final focal direction arrow starts from the final same-sized black dot.
    final_box = payload.main_gt[payload.main_frames[-1]]
    final_direction = _motion_unit_direction(
        payload.main_frames,
        [payload.main_gt[f] for f in payload.main_frames],
        fallback_yaw=float(final_box[6]),
        lookback_frames=max(2, int(round(0.5 * FPS))),
    )
    ax.arrow(
        float(final_box[0]),
        float(final_box[1]),
        float(final_direction[0] * MAIN_FINAL_ARROW_LENGTH_M),
        float(final_direction[1] * MAIN_FINAL_ARROW_LENGTH_M),
        width=0.018,
        head_width=MAIN_FINAL_ARROW_HEAD_WIDTH_M,
        head_length=MAIN_FINAL_ARROW_HEAD_LENGTH_M,
        length_includes_head=True,
        fc="black",
        ec="black",
        linewidth=MAIN_FINAL_LINEWIDTH,
        zorder=21,
    )

    # Simplified non-overlapping visual regimes. Single-row is cleaner and makes
    # the mutually exclusive schematic segmentation immediately obvious.
    scene_top = payload.scene_ylim[1]
    if REGIME_ARROW_LAYOUT == "single_row":
        row_y = {key: scene_top + 0.38 for key in REGIME_COLORS}
        regime_top = scene_top + 0.68
    elif REGIME_ARROW_LAYOUT == "stacked":
        row_gap = 0.42
        row_y = {
            "gap_coasting": scene_top + 0.26,
            "continuous": scene_top + 0.26 + row_gap,
            "initialization": scene_top + 0.26 + 2 * row_gap,
        }
        regime_top = row_y["initialization"] + 0.28
    else:
        raise ValueError("REGIME_ARROW_LAYOUT must be 'single_row' or 'stacked'")

    gt_x = {f: float(payload.main_gt[f][0]) for f in payload.main_frames}
    for key in ("initialization", "continuous", "gap_coasting"):
        for start, end in payload.regimes.get(key, []):
            available = [f for f in payload.main_frames if start <= f <= end]
            if available:
                _draw_regime_arrow(
                    ax,
                    gt_x[available[0]],
                    gt_x[available[-1]],
                    row_y[key],
                    REGIME_COLORS[key],
                )

    ax.set_xlim(payload.xlim)
    ax.set_ylim(payload.scene_ylim[0], regime_top)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    if SHOW_FIG1_TITLE:
        ax.set_title(f"{sequence} — GT ID {gt_id}")
    despine(ax)

    from matplotlib.patches import Rectangle
    from matplotlib.lines import Line2D

    legend_handles = [
        # GT boxes as rectangle proxy
        Rectangle(
            (0, 0), 1, 1,
            fill=False,
            edgecolor=GT_COLOR,
            linewidth=2.0,
            label="GT boxes",
        ),

        # Matched detections as rectangle proxy
        Rectangle(
            (0, 0), 1, 1,
            fill=False,
            edgecolor=DET_COLOR,
            linewidth=2.0,
            label="Matched detections",
        ),

        # Main pedestrian dots
        Line2D(
            [0], [0],
            marker="o",
            color=MAIN_DOT_COLOR,
            markerfacecolor=MAIN_DOT_COLOR,
            markeredgecolor=MAIN_DOT_COLOR,
            markersize=7,
            linestyle="None",
            label="Pedestrian trajectory",
        ),
    ]

    if SHOW_OTHER_PEDESTRIANS:
        legend_handles.append(
            Line2D(
                [0], [0],
                marker="o",
                color=CONTEXT_COLOR,
                markerfacecolor=CONTEXT_COLOR,
                markeredgecolor=CONTEXT_COLOR,
                markersize=8,
                linestyle="None",
                label="Other pedestrians (GT)",
            )
        )

    region_handles = [
        Line2D(
            [0, 1], [0, 0],
            color=REGIME_COLORS["initialization"],
            lw=2.0,
            marker=">",
            markevery=[1],
            markersize=7.0,
            label="Initialize",
        ),
        Line2D(
            [0, 1], [0, 0],
            color=REGIME_COLORS["continuous"],
            lw=2.0,
            marker=">",
            markevery=[1],
            markersize=7.0,
            label="Continuous tracking",
        ),
        Line2D(
            [0, 1], [0, 0],
            color=REGIME_COLORS["gap_coasting"],
            lw=2.0,
            marker=">",
            markevery=[1],
            markersize=7.0,
            label="Gap coasting",
        ),
    ]

    # Separate legend rows keep the figure easier to parse and allow the context
    # entry to disappear completely when SHOW_OTHER_PEDESTRIANS=False.
    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.98),
        ncol=len(legend_handles),
        frameon=False,
        handlelength=2.2,
        columnspacing=1.8,
    )
    fig.legend(
        handles=region_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.92),
        ncol=3,
        frameon=False,
        handlelength=2.2,
        columnspacing=1.5,
    )
    fig.subplots_adjust(top=0.79, left=0.085, right=0.99, bottom=0.16)
    return fig


In [ ]:
# Cell 8 — Build and save Figure 1
payload = prepare_figure1_payload(
    sequence=SEQUENCE,
    selected_gt_id=SELECTED_GT_ID,
    frame_start=FRAME_START,
    frame_end=FRAME_END,
)

if VERIFY_GLOBAL_ALIGNMENT and len(payload.alignment_errors_m):
    median_err = float(np.median(payload.alignment_errors_m))
    p95_err = float(np.quantile(payload.alignment_errors_m, 0.95))
    print(
        "Local GT + SteamLO -> stored global GT alignment: "
        f"median={median_err:.4f} m, P95={p95_err:.4f} m"
    )
    if p95_err > GLOBAL_ALIGNMENT_WARN_M:
        warnings.warn(
            f"Global alignment P95 ({p95_err:.3f} m) exceeds "
            f"{GLOBAL_ALIGNMENT_WARN_M:.3f} m. Check whether gt__global "
            "was produced from the same SteamLO odometry."
        )

print("Simplified non-overlapping visual regime spans:")
for key, spans in payload.regimes.items():
    print(f"  {key}: {spans}")

if PRINT_CONTEXT_DIAGNOSTICS and SHOW_OTHER_PEDESTRIANS:
    print(
        "Context plotting diagnostics:\n"
        f"  temporal window: frames {payload.context_window[0]}-{payload.context_window[1]} "
        f"({(payload.context_window[1] - payload.context_window[0] + 1) / FPS:.2f} s)\n"
        f"  other GT tracks in temporal window: {payload.context_total_tracks}\n"
        f"  tracks with >=1 state inside plotted XY limits: {payload.context_visible_tracks}\n"
        f"  tracks whose actual latest state is inside limits: {payload.context_latest_visible_tracks}\n"
        f"  minimum same-frame focal->other distance in window: "
        f"{payload.context_min_same_frame_distance_m:.2f} m\n"
        f"  nearest other pedestrian at focal final frame: "
        f"{payload.context_final_frame_nearest_distance_m:.2f} m"
    )

fig1 = plot_figure1(payload, SEQUENCE, SELECTED_GT_ID)
fig1_path = FIGURE_OUTPUT_DIR / (
    f"figure1_jrdb_regimes__{SEQUENCE}__gtid{SELECTED_GT_ID}.pdf"
)
if fig1_path.exists() and not OVERWRITE_FIGURES:
    print(f"Figure exists; not overwriting: {fig1_path}")
else:
    fig1.savefig(fig1_path)
    print(f"Saved: {fig1_path}")
plt.show()


In [ ]:
# Cell 9 — Moderate pseudo-detection corruption helpers

def _clip01(x: float) -> float:
    return float(max(0.0, min(1.0, float(x))))


def _ceil_frames(seconds: float, fps: float) -> int:
    return max(1, int(math.ceil(float(seconds) * float(fps))))


def _wrap_angle_rad_pi(a: float) -> float:
    return (float(a) + math.pi) % (2.0 * math.pi) - math.pi


def _safe_pos(x: float, eps: float = 1e-3) -> float:
    return float(max(eps, float(x)))


def _trunc_normal(rng, mu, sigma, size, n_sigma=2.0, max_tries=50, dtype=np.float32):
    mu, sigma = float(mu), float(sigma)
    if sigma <= 0:
        return np.full(size, mu, dtype=dtype)
    lo, hi = mu - n_sigma*sigma, mu + n_sigma*sigma
    out = rng.normal(mu, sigma, size=size).astype(dtype, copy=False)
    mask = (out < lo) | (out > hi)
    tries = 0
    while np.any(mask) and tries < max_tries:
        out[mask] = rng.normal(mu, sigma, size=int(mask.sum())).astype(dtype, copy=False)
        mask = (out < lo) | (out > hi)
        tries += 1
    if np.any(mask):
        out = np.clip(out, lo, hi).astype(dtype, copy=False)
    return out


def _set_height_keep_bottom(box7: np.ndarray, new_h: float) -> None:
    bottom = float(box7[2] - 0.5 * box7[5])
    h = _safe_pos(new_h)
    box7[5] = h
    box7[2] = bottom + 0.5 * h


PSEUDO_L2 = {
    "dropout_p_start": 0.020,
    "dropout_min_s": 0.05,
    "dropout_max_s": 2.00,
    "instability_k_modes": 4,
    "instability_p_switch": 0.10,
    "instability_mode_xy_sigma_m": 0.06,
    "instability_jitter_xy_sigma_m": 0.02,
    "instability_mode_yaw_sigma_rad": 0.80,
    "instability_jitter_yaw_sigma_rad": 0.15,
    "instability_mode_lwh_sigma_rel": 0.05,
    "instability_jitter_lwh_sigma_rel": 0.02,
}


def make_dropout_keep_mask(frames, fps, p_start, min_s, max_s, rng):
    frames = np.asarray(frames, dtype=int)
    keep = np.ones(len(frames), dtype=bool)
    minimum = _ceil_frames(min_s, fps)
    maximum = max(minimum, _ceil_frames(max_s, fps))
    dropout_until = -10**9
    for index, frame in enumerate(frames.tolist()):
        if frame <= dropout_until:
            keep[index] = False
        elif rng.random() < _clip01(p_start):
            duration = int(rng.integers(minimum, maximum + 1))
            dropout_until = int(frame) + duration - 1
            keep[index] = False
    return keep


def apply_instability(frames, gt_boxes, rng, p=PSEUDO_L2):
    boxes = np.asarray(gt_boxes, dtype=np.float32).copy()
    n = len(boxes)
    K = int(p["instability_k_modes"])
    mode_xy = _trunc_normal(
        rng, 0.0, p["instability_mode_xy_sigma_m"], size=(K, 2)
    )
    mode_yaw = _trunc_normal(
        rng, 0.0, p["instability_mode_yaw_sigma_rad"], size=(K,)
    )
    mode_lwh = _trunc_normal(
        rng, 0.0, p["instability_mode_lwh_sigma_rel"], size=(K, 3)
    )
    mode_index = int(rng.integers(0, K))

    for i in range(n):
        if i > 0 and K > 1 and rng.random() < _clip01(p["instability_p_switch"]):
            new_index = int(rng.integers(0, K - 1))
            if new_index >= mode_index:
                new_index += 1
            mode_index = new_index
        boxes[i, 0:2] += mode_xy[mode_index]
        boxes[i, 6] = _wrap_angle_rad_pi(
            float(boxes[i, 6]) + float(mode_yaw[mode_index])
        )
        rel = mode_lwh[mode_index]
        boxes[i, 3] = _safe_pos(float(boxes[i, 3]) * (1.0 + float(rel[0])))
        boxes[i, 4] = _safe_pos(float(boxes[i, 4]) * (1.0 + float(rel[1])))
        _set_height_keep_bottom(
            boxes[i],
            _safe_pos(float(boxes[i, 5]) * (1.0 + float(rel[2]))),
        )

        boxes[i, 0] += float(_trunc_normal(
            rng, 0.0, p["instability_jitter_xy_sigma_m"], size=()
        ))
        boxes[i, 1] += float(_trunc_normal(
            rng, 0.0, p["instability_jitter_xy_sigma_m"], size=()
        ))
        jitter = _trunc_normal(
            rng, 0.0, p["instability_jitter_lwh_sigma_rel"], size=(3,)
        )
        boxes[i, 3] = _safe_pos(float(boxes[i, 3]) * (1.0 + float(jitter[0])))
        boxes[i, 4] = _safe_pos(float(boxes[i, 4]) * (1.0 + float(jitter[1])))
        _set_height_keep_bottom(
            boxes[i],
            _safe_pos(float(boxes[i, 5]) * (1.0 + float(jitter[2]))),
        )
        boxes[i, 6] = _wrap_angle_rad_pi(
            float(boxes[i, 6])
            + float(_trunc_normal(
                rng, 0.0, p["instability_jitter_yaw_sigma_rad"], size=()
            ))
        )
    return boxes.astype(float)


def generate_synthetic_variants(n_frames=300, fps=15.0, seed=101):
    frames = np.arange(n_frames, dtype=int)
    t = frames / float(fps)
    x0, y0, speed = -3.0, -1.0, 0.35
    l, w, h = 0.60, 0.40, 1.70
    clean = np.stack([
        x0 + speed*t,
        np.full_like(t, y0),
        np.zeros_like(t),
        np.full_like(t, l),
        np.full_like(t, w),
        np.full_like(t, h),
        np.zeros_like(t),
    ], axis=1).astype(float)

    # Generate the two corruption components ONCE. The individual rows show
    # each component, while Combined is literally their composition:
    #   Combined = instability realization restricted by the same dropout mask.
    dropout_rng = np.random.default_rng(seed+40)
    instability_rng = np.random.default_rng(seed+50)

    keep_dropout = make_dropout_keep_mask(
        frames,
        fps,
        PSEUDO_L2["dropout_p_start"],
        PSEUDO_L2["dropout_min_s"],
        PSEUDO_L2["dropout_max_s"],
        dropout_rng,
    )
    instability = apply_instability(
        frames,
        clean,
        instability_rng,
    )

    return {
        "Clean": (frames, clean),
        "Dropout": (frames[keep_dropout], clean[keep_dropout]),
        "Instability": (frames, instability),
        "Combined": (frames[keep_dropout], instability[keep_dropout]),
    }


In [ ]:
# Cell 10 — Figure 2: faded-marker history + final black box
PSEUDO_SEED = 9485941
def _draw_final_bbox_black(ax, box: np.ndarray) -> None:
    corners = _bev_corners_ccw(box)
    ax.add_patch(Polygon(
        corners,
        closed=True,
        fill=False,
        edgecolor="black",
        linewidth=PSEUDO_FINAL_BBOX_LW,
        zorder=8,
    ))


def plot_pseudo_failure_modes() -> plt.Figure:
    # Axis-free visualization with bone_r time coloring,
    # faded point history, only the final/current state as a black bbox.
    apply_scai_white_style()
    variants = generate_synthetic_variants(
        PSEUDO_N_FRAMES,
        PSEUDO_FPS,
        PSEUDO_SEED,
    )
    fig, axes = plt.subplots(
        4,
        1,
        dpi=300,
        figsize=PSEUDO_FIG_SIZE,
    )

    all_boxes = np.vstack([
        boxes for _, boxes in variants.values() if len(boxes)
    ])
    xmin = float(all_boxes[:, 0].min() - 0.75)
    xmax = float(all_boxes[:, 0].max() + 0.75)
    ymin = float(all_boxes[:, 1].min() - 0.65)
    ymax = float(all_boxes[:, 1].max() + 0.65)

    for ax, (title, (frames, boxes)) in zip(axes, variants.items()):
        ax.scatter(
            boxes[:, 0],
            boxes[:, 1],
            c=frames.astype(float),
            cmap=PSEUDO_COLORMAP,
            s=PSEUDO_MARKER_SIZE_PT2,
            edgecolors="none",
            alpha=PSEUDO_MARKER_ALPHA,
            zorder=3,
        )
        if len(boxes):
            _draw_final_bbox_black(ax, boxes[-1])
        ax.text(
            0.0,
            1.02,
            title,
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=9.0,
            fontweight="bold",
            color=SCAI["text"],
        )
        if title == "Clean" and PSEUDO_SHOW_PEDESTRIAN_LABEL and len(boxes):
            ax.text(
                float(boxes[-1, 0] - 0.55),
                float(boxes[-1, 1] + 0.42),
                "Pedestrian",
                ha="center",
                va="bottom",
                fontsize=8.0,
                color=SCAI["text_dim"],
            )
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect("equal", adjustable="box")
        ax.axis("off")

    fig.subplots_adjust(
        left=0.08,
        right=0.98,
        top=0.88,
        bottom=0.03,
        hspace=0.42,
    )
    return fig


fig2 = plot_pseudo_failure_modes()
fig2_path = FIGURE_OUTPUT_DIR / "figure2_pseudo_detection_failure_modes.pdf"
if fig2_path.exists() and not OVERWRITE_FIGURES:
    print(f"Figure exists; not overwriting: {fig2_path}")
else:
    fig2.savefig(fig2_path)
    print(f"Saved: {fig2_path}")
plt.show()


## Practical workflow

1. Run Cells 1–6. The candidate scan and per-sequence GT↔detector matches are cached automatically; rerunning with `OVERWRITE_CACHE=False` reuses compatible saved results.
2. Inspect the complete candidate DataFrame printed in Cell 6 and copy a preferred `sequence`, `gt_id`, and optional frame range into Cell 1.
3. Run Cells 7–8 to generate Figure 1. `SHOW_OTHER_PEDESTRIANS=False` removes context pedestrians **and** their legend entry.
4. Use `REGIME_OVERRIDE` only if publication-level manual control is needed. The automatic current arrows are deliberately non-overlapping.
5. Figure 2 is independent of the JRDB paths and can be generated from Cells 9–10.

### Candidate criteria implemented in current

- detector support ≥ 50%,
- at least two internally bounded detector gaps with >3 consecutive misses,
- **at least one of those gaps is between 0.75 and 2.0 s**, inclusive,
- trajectory duration ≥ 5 s,
- global-GT smoothed speed >0.4 m/s for ≥95% of valid trajectory samples.

### Figure 1 context rule

Other-pedestrian GT states are considered only over the last 10 s ending at the focal pedestrian's selected final frame (or the entire selected interval if it is shorter). Their complete temporal tracks are retained, while out-of-bounds states are clipped only at draw time. An arrow is drawn only if the pedestrian's **actual latest state** in that temporal window lies inside the displayed XY limits.
